# Stroke Prediction: Data Visualization & Preprocessing

Prepare the Kaggle healthcare stroke dataset for analysis. This notebook explores the observed data, cleans it, encodes features, splits the data, scales numerical features, and balances **training data only**. It does not train a diagnostic model or provide medical advice.

## 1. Load and inspect the data

Place `healthcare-dataset-stroke-data.csv` in the same folder from which this notebook is launched. The loader checks the current working directory and gives its exact path if the file is missing.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

sns.set_theme(style="whitegrid")

DATA_FILE = Path.cwd() / "healthcare-dataset-stroke-data.csv"
if not DATA_FILE.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_FILE}\n"
        "Place healthcare-dataset-stroke-data.csv in the notebook's current working directory and rerun."
    )

df = pd.read_csv(DATA_FILE)
required_columns = {
    "stroke", "gender", "work_type", "smoking_status", "ever_married",
    "Residence_type", "age", "avg_glucose_level", "bmi", "id",
    "hypertension", "heart_disease",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns from {DATA_FILE}")
display(df.head())

In [ ]:
df.info()
display(df.describe(include="all").T)

## 2. Exploratory data analysis

Remove the single `Other` gender row requested in the assignment before plotting. The code reports how many such rows were found rather than assuming the dataset version.

In [ ]:
other_gender_count = int(df["gender"].eq("Other").sum())
print(f"Rows with gender='Other': {other_gender_count}")
df = df.loc[~df["gender"].eq("Other")].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.countplot(data=df, x="stroke", ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Stroke class distribution")
axes[0, 0].set_xlabel("Stroke (0 = No, 1 = Yes)")

for ax, column in zip(axes.flat[1:], ["gender", "work_type", "smoking_status"]):
    sns.countplot(data=df, x=column, hue="stroke", ax=ax)
    ax.set_title(f"{column.replace('_', ' ').title()} by stroke status")
    ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = ["age", "avg_glucose_level", "bmi"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, column in zip(axes, numeric_columns):
    sns.histplot(data=df, x=column, hue="stroke", kde=True, stat="density",
                 common_norm=False, element="step", ax=ax)
    ax.set_title(f"Distribution of {column.replace('_', ' ')}")
plt.tight_layout()
plt.show()

In [ ]:
correlation_columns = [
    "age", "hypertension", "heart_disease", "avg_glucose_level", "bmi", "stroke"
]
plt.figure(figsize=(9, 6))
sns.heatmap(df[correlation_columns].corr(numeric_only=True), annot=True,
            cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation of selected numeric features")
plt.tight_layout()
plt.show()

## 3. Clean the data

Impute missing BMI values with the median, remove exact duplicate rows, and drop the identifier. High glucose values are inspected with a boxplot; they are not automatically discarded because extreme measurements can be valid and clinically relevant.

In [ ]:
print("Missing values before cleaning:")
display(df.isnull().sum().sort_values(ascending=False))

bmi_median = df["bmi"].median()
if pd.isna(bmi_median) and df["bmi"].isna().any():
    raise ValueError("BMI is entirely missing; its median cannot be used for imputation.")
df["bmi"] = df["bmi"].fillna(bmi_median)

duplicate_count = int(df.duplicated().sum())
print(f"Exact duplicate rows removed: {duplicate_count}")
df = df.drop_duplicates().copy()
if "id" in df.columns:
    df = df.drop(columns="id")
df = df.reset_index(drop=True)

print("Missing values after BMI imputation:")
display(df.isnull().sum().sort_values(ascending=False))
if df.isnull().any().any():
    raise ValueError("Missing values remain; inspect the table above and clean those columns before continuing.")

In [ ]:
plt.figure(figsize=(9, 3))
sns.boxplot(x=df["avg_glucose_level"], color="cornflowerblue")
plt.title("Average glucose level: outlier inspection")
plt.tight_layout()
plt.show()

q1 = df["avg_glucose_level"].quantile(0.25)
q3 = df["avg_glucose_level"].quantile(0.75)
iqr = q3 - q1
outlier_mask = (df["avg_glucose_level"] < q1 - 1.5 * iqr) | (
    df["avg_glucose_level"] > q3 + 1.5 * iqr
)
print(f"Glucose values outside the 1.5-IQR range: {int(outlier_mask.sum()):,}")
print("These observations are retained: an IQR flag alone does not establish a data error.")

## 4. Encode features and split the data

One-hot encode categorical columns, then split with stratification so the rare stroke class is represented in both sets. The target is kept separate from the features.

In [ ]:
categorical_columns = [
    "gender", "work_type", "smoking_status", "ever_married", "Residence_type"
]
df = pd.get_dummies(df, columns=categorical_columns, dtype=int)
df = df.reset_index(drop=True)

X = df.drop(columns="stroke")
y = df["stroke"].astype(int)
if y.nunique() != 2:
    raise ValueError(f"Expected a binary stroke target, found values: {sorted(y.unique())}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)
print("Training target proportions:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion"))

## 5. Scale numeric features

Fit `StandardScaler` on the training partition only and use that fitted scaler to transform the test partition. This prevents test-set statistics from leaking into preprocessing.

In [ ]:
scale_columns = [column for column in numeric_columns if column in X_train.columns]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled.loc[:, scale_columns] = scaler.fit_transform(X_train[scale_columns])
X_test_scaled.loc[:, scale_columns] = scaler.transform(X_test[scale_columns])

print("Scaled columns:", scale_columns)
print("Training scaled means (approximately zero):")
display(X_train_scaled[scale_columns].mean().rename("mean"))

## 6. Handle class imbalance on training data only

Stroke cases are usually a minority. Randomly oversample the minority class in the **training** partition only. Keep the original test set unchanged so it remains representative for later evaluation.

In [ ]:
print("Training counts before oversampling:")
display(y_train.value_counts().sort_index().rename("count"))

train_frame = X_train_scaled.copy()
train_frame["stroke"] = y_train.to_numpy()
class_counts = train_frame["stroke"].value_counts()
if len(class_counts) < 2:
    raise ValueError("Both target classes must be present in the training partition.")

majority_class = class_counts.idxmax()
minority_class = class_counts.idxmin()
majority_rows = train_frame.loc[train_frame["stroke"] == majority_class]
minority_rows = train_frame.loc[train_frame["stroke"] == minority_class]
minority_upsampled = resample(
    minority_rows,
    replace=True,
    n_samples=len(majority_rows),
    random_state=42,
)
balanced_train = pd.concat([majority_rows, minority_upsampled], ignore_index=True)
balanced_train = balanced_train.sample(frac=1, random_state=42).reset_index(drop=True)

X_train_balanced = balanced_train.drop(columns="stroke")
y_train_balanced = balanced_train["stroke"].astype(int)
print("Training counts after oversampling:")
display(y_train_balanced.value_counts().sort_index().rename("count"))
print("Test counts (unchanged):")
display(y_test.value_counts().sort_index().rename("count"))

## 7. Summary and hypothetical patient persona

Review the dataset-derived summaries below. The persona is an illustrative description of factors to explore in this dataset—not a clinical risk score, diagnosis, or substitute for professional care.

In [ ]:
print(f"Stroke prevalence in the cleaned dataset: {y.mean():.2%}")
print("Median numeric characteristics by observed stroke label:")
display(
    df.groupby("stroke")[["age", "avg_glucose_level", "bmi", "hypertension", "heart_disease"]]
      .median()
      .rename(index={0: "No stroke", 1: "Stroke"})
)
print(
    f"Prepared {len(X_train_balanced):,} balanced training rows and "
    f"{len(X_test_scaled):,} untouched test rows across {X.shape[1]} encoded features."
)

### Hypothetical high-risk profile

A fictional older adult with a history of hypertension or heart disease and elevated average glucose could be used as a **discussion persona** when interpreting this dataset. The dataset contains population-level associations; it cannot establish that these characteristics cause stroke or predict an individual patient's outcome. Actual symptoms or personal risk concerns require assessment by a qualified healthcare professional.